### **IMPORTING LIBRARIES AND SETTINGS**

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Machine Learning libraries
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, roc_curve, precision_recall_curve
)
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


# Visualization settings
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8')
sns.set_theme(style='whitegrid', palette='bright')

print("All libraries imported successfully")
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

All libraries imported successfully
Analysis Date: 2026-05-29 21:49:32


### **LOADING THE DATASET**

This task focuses on loading the raw CSV dataset into a Jupyter Notebook environment and performing a comprehensive initial investigation of the dataset. The goal is to understand the structure, schema, and statistical properties of the hydraulic sensor data used at Bosch Rexroth AG. This step is critical because it establishes foundational awareness of what variables are available, how the data is organized, and whether there are any immediate quality concerns before deeper analysis begins.

##### **Task**
- Loading the CSV dataset into Jupyter Notebook using pandas
- Inspecting dataset dimensions (rows and columns)
- Identifying data types of each feature
- Displaying sample records (head, tail, random samples)
- Listing all available features and categorize them:

  - Sensor variables

  - Target variables

  - Metadata (if any)

- Generating the summary statistics (mean, std, min, max)
- Identifying potential data quality issues
- Documenting the initial dataset understanding

##### **Inspecting the data on all four tables**

In [ ]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
sensor_df = pd.read_csv("/content/drive/MyDrive/predictive maintenance project/sensor_telemetry_cleaneed.csv")
maintenance_df  = pd.read_csv("/content/drive/MyDrive/predictive maintenance project/maintenance_log.csv")
equipment_df  = pd.read_csv("/content/drive/MyDrive/predictive maintenance project/equipment_master.csv")
failure_df   = pd.read_csv("/content/drive/MyDrive/predictive maintenance project/failure_labels.csv")


In [ ]:
# Creating a dataset dictionary for group inspection
dataset_dict = {"sensor_df": sensor_df, "maintenance_df": maintenance_df, "equipment_df": equipment_df, "failure_df": failure_df}
#inspecting the dataset by checking the first five rows
for name, df in dataset_dict.items():
    print('=' * 60)
    print(f"dataset: {name}\n")
    display(df.head())


dataset: sensor_df



,timestamp,machine_id,pressure_bar,temp_celsius,flow_lpm,vibration_x_g,vibration_y_g,pump_rpm,is_anomaly,failure_mode,rul_hours,is_sensor_dropout,shift,day_of_week
0,01/02/2024 03:15,HPU_01,128.10,51.69,85.60,0.0665,0.0665,1472.0,1,pump_wear,332.8,0,Night,3
1,01/02/2024 03:16,HPU_01,126.68,52.33,87.94,0.0799,0.0799,1471.0,1,pump_wear,332.7,0,Night,3
2,01/02/2024 03:17,HPU_01,124.49,51.79,88.07,0.0948,0.0948,1468.0,1,pump_wear,332.7,0,Night,3
3,01/02/2024 03:18,HPU_01,124.14,51.80,89.51,0.1232,0.1232,1471.0,1,pump_wear,332.7,0,Night,3
4,01/02/2024 03:19,HPU_01,124.88,50.97,86.94,0.0515,0.0515,1473.0,1,pump_wear,332.7,0,Night,3


dataset: maintenance_df



,maintenance_id,machine_id,action_timestamp,action_type,component_replaced,technician_id,cost_usd
0,M_026,HPU_05,03/10/2023 17:09,Reactive,Valve,T_001,7840
1,M_009,HPU_02,04/10/2023 20:06,Preventive,Filter,T_002,1896
2,M_036,HPU_08,06/10/2023 07:26,Preventive,Seal,T_004,738
3,M_023,HPU_05,07/10/2023 07:47,Preventive,Oil,T_001,1476
4,M_015,HPU_03,10/10/2023 06:41,Preventive,Seal,T_008,707


dataset: equipment_df



,machine_id,installation_date,total_operating_hours,fluid_type,last_filter_change_date,maintenance_priority
0,HPU_01,2023-03-13,14012,mineral_oil,2023-12-19,Low
1,HPU_02,2022-10-28,9572,mineral_oil,2023-11-24,Medium
2,HPU_03,2023-03-23,7848,mineral_oil,2023-10-14,Low
3,HPU_04,2022-04-29,5976,mineral_oil,2023-12-18,High
4,HPU_05,2023-04-01,12623,mineral_oil,2023-11-25,Low


dataset: failure_df



,failure_event_id,machine_id,failure_timestamp,failure_mode,degradation_start_timestamp,repair_cost_usd,downtime_hours
0,F_001,HPU_01,2024-02-15,pump_wear,2024-02-01,25094,13.2
1,F_002,HPU_02,2024-02-22,valve_leakage,2024-02-12,8711,9.8
2,F_003,HPU_03,2024-02-08,contamination,2024-02-03,24507,6.3
3,F_004,HPU_04,2024-02-28,pump_wear,2024-02-16,29482,22.4
4,F_005,HPU_05,2024-01-31,valve_leakage,2024-01-21,14065,8.2


##### **Checking the dimension of the datasets (all four dataset)**

In [ ]:
# checking dataset dimension
print("=" * 60)
print("DATASET DIMENSION")
print("=" * 60)
for name, dataset in dataset_dict.items():
    print(f"  {name:<22}: {dataset.shape[0]:>8,} rows × {dataset.shape[1]} cols")
print("=" * 60)

DATASET DIMENSION
  sensor_df             :  864,000 rows × 14 cols
  maintenance_df        :       51 rows × 7 cols
  equipment_df          :       10 rows × 6 cols
  failure_df            :        9 rows × 7 cols


##### **Checking the data information for all the datasets**

In [ ]:
# Dataset information check
for name, df in dataset_dict.items():
    print('=' * 60)
    print(f"dataset: {name}\n")
    # print('=' * 60)
    display(df.info())


dataset: sensor_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 864000 entries, 0 to 863999
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   timestamp          864000 non-null  object 
 1   machine_id         864000 non-null  object 
 2   pressure_bar       861410 non-null  float64
 3   temp_celsius       861410 non-null  float64
 4   flow_lpm           861410 non-null  float64
 5   vibration_x_g      861410 non-null  float64
 6   vibration_y_g      861410 non-null  float64
 7   pump_rpm           861410 non-null  float64
 8   is_anomaly         864000 non-null  int64  
 9   failure_mode       864000 non-null  object 
 10  rul_hours          864000 non-null  float64
 11  is_sensor_dropout  864000 non-null  int64  
 12  shift              864000 non-null  object 
 13  day_of_week        864000 non-null  int64  
dtypes: float64(7), int64(3), object(4)
memory usage: 92.3+ MB


None

dataset: maintenance_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   maintenance_id      51 non-null     object
 1   machine_id          51 non-null     object
 2   action_timestamp    51 non-null     object
 3   action_type         51 non-null     object
 4   component_replaced  42 non-null     object
 5   technician_id       51 non-null     object
 6   cost_usd            51 non-null     int64 
dtypes: int64(1), object(6)
memory usage: 2.9+ KB


None

dataset: equipment_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   machine_id               10 non-null     object
 1   installation_date        10 non-null     object
 2   total_operating_hours    10 non-null     int64 
 3   fluid_type               10 non-null     object
 4   last_filter_change_date  10 non-null     object
 5   maintenance_priority     10 non-null     object
dtypes: int64(1), object(5)
memory usage: 612.0+ bytes


None

dataset: failure_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   failure_event_id             9 non-null      object 
 1   machine_id                   9 non-null      object 
 2   failure_timestamp            9 non-null      object 
 3   failure_mode                 9 non-null      object 
 4   degradation_start_timestamp  9 non-null      object 
 5   repair_cost_usd              9 non-null      int64  
 6   downtime_hours               9 non-null      float64
dtypes: float64(1), int64(1), object(5)
memory usage: 636.0+ bytes


None

##### **Checking for missing values in all four datasets**

In [ ]:
# Check for missing values
for name, df in dataset_dict.items():
    print('=' * 60)
    print(f"dataset: {name}\n")
    display(df.isna().sum())


dataset: sensor_df



,0
timestamp,0
machine_id,0
pressure_bar,2590
temp_celsius,2590
flow_lpm,2590
vibration_x_g,2590
vibration_y_g,2590
pump_rpm,2590
is_anomaly,0
failure_mode,0


dataset: maintenance_df



,0
maintenance_id,0
machine_id,0
action_timestamp,0
action_type,0
component_replaced,9
technician_id,0
cost_usd,0


dataset: equipment_df



,0
machine_id,0
installation_date,0
total_operating_hours,0
fluid_type,0
last_filter_change_date,0
maintenance_priority,0


dataset: failure_df



,0
failure_event_id,0
machine_id,0
failure_timestamp,0
failure_mode,0
degradation_start_timestamp,0
repair_cost_usd,0
downtime_hours,0


##### **Checking for duplicated datas in the datasets**

In [ ]:
# Check for duplicates
for name, df in dataset_dict.items():
    print('=' * 60)
    print(f"dataset: {name}")
    display(df.duplicated().sum())


dataset: sensor_df


np.int64(0)

dataset: maintenance_df


np.int64(0)

dataset: equipment_df


np.int64(0)

dataset: failure_df


np.int64(0)

##### **Checking for datatypes**

In [ ]:
# Check for datatypes
for name, df in dataset_dict.items():
    print('=' * 60)
    print(f"dataset: {name}")
    display(df.dtypes)


dataset: sensor_df


,0
timestamp,object
machine_id,object
pressure_bar,float64
temp_celsius,float64
flow_lpm,float64
vibration_x_g,float64
vibration_y_g,float64
pump_rpm,float64
is_anomaly,int64
failure_mode,object


dataset: maintenance_df


,0
maintenance_id,object
machine_id,object
action_timestamp,object
action_type,object
component_replaced,object
technician_id,object
cost_usd,int64


dataset: equipment_df


,0
machine_id,object
installation_date,object
total_operating_hours,int64
fluid_type,object
last_filter_change_date,object
maintenance_priority,object


dataset: failure_df


,0
failure_event_id,object
machine_id,object
failure_timestamp,object
failure_mode,object
degradation_start_timestamp,object
repair_cost_usd,int64
downtime_hours,float64


##### **Checking for unique values in all the datasets**

In [ ]:
# Check for unique values
for name, df in dataset_dict.items():
    print('=' * 60)
    print(f"dataset: {name}")
    display(df.nunique())


dataset: sensor_df


,0
timestamp,86400
machine_id,10
pressure_bar,9509
temp_celsius,1916
flow_lpm,5526
vibration_x_g,12925
vibration_y_g,12922
pump_rpm,100
is_anomaly,2
failure_mode,5


dataset: maintenance_df


,0
maintenance_id,51
machine_id,10
action_timestamp,51
action_type,3
component_replaced,5
technician_id,8
cost_usd,50


dataset: equipment_df


,0
machine_id,10
installation_date,9
total_operating_hours,10
fluid_type,2
last_filter_change_date,9
maintenance_priority,3


dataset: failure_df


,0
failure_event_id,9
machine_id,9
failure_timestamp,9
failure_mode,4
degradation_start_timestamp,8
repair_cost_usd,9
downtime_hours,9


##### **Checking for Summary statistics on all four datasets**

In [ ]:
# Check for summary statistics
for name, df in dataset_dict.items():
    print('=' * 60)
    print(f"dataset: {name}\n")
    display(df.describe())


dataset: sensor_df



,pressure_bar,temp_celsius,flow_lpm,vibration_x_g,vibration_y_g,pump_rpm,is_anomaly,rul_hours,is_sensor_dropout,day_of_week
count,861410.000000,861410.000000,861410.000000,861410.000000,861410.000000,861410.000000,864000.000000,864000.000000,864000.000000,864000.000000
mean,123.745802,52.218564,86.145165,0.143306,0.142908,1479.726617,0.146667,323.167889,0.002998,2.900000
std,16.610808,2.281346,10.850925,0.365332,0.365400,68.261889,0.353774,226.183963,0.054669,1.989138
min,42.960000,-9.940000,31.250000,0.050000,0.050000,1000.000000,0.000000,0.000000,0.000000,0.000000
25%,113.000000,51.080000,77.280000,0.051500,0.051000,1463.000000,0.000000,8.000000,0.000000,1.000000
50%,125.920000,52.170000,88.170000,0.089900,0.089600,1480.000000,0.000000,500.000000,0.000000,3.000000
75%,136.620000,53.120000,94.870000,0.147000,0.146500,1518.000000,0.000000,500.000000,0.000000,5.000000
max,598.470000,64.860000,108.130000,14.739100,14.965300,1540.000000,1.000000,500.000000,1.000000,6.000000


dataset: maintenance_df



,cost_usd
count,51.000000
mean,1921.607843
std,1745.885839
min,306.000000
25%,801.500000
50%,1268.000000
75%,2330.000000
max,7840.000000


dataset: equipment_df



,total_operating_hours
count,10.000000
mean,12329.100000
std,5820.897343
min,5869.000000
25%,8279.000000
50%,11427.000000
75%,14081.750000
max,24309.000000


dataset: failure_df



,repair_cost_usd,downtime_hours
count,9.000000,9.000000
mean,20366.222222,11.855556
std,7878.436961,4.766841
min,8711.000000,6.300000
25%,14065.000000,8.700000
50%,23021.000000,10.500000
75%,25094.000000,13.200000
max,30336.000000,22.400000


The predictive maintenance project is built on four interconnected datasets containing sensor telemetry, failure events, maintenance history, and equipment metadata for hydraulic systems. An initial data assessment was conducted to understand the structure.

1. Sensor Telemetry Dataset (sensor_df)

The sensor_df dataset is the largest and most important dataset in the project, containing 864,000 records and 14 features representing high-frequency time-series sensor telemetry from hydraulic machines. The dataset includes operational parameters such as hydraulic pressure, oil temperature, flow rate, vibration measurements, and pump rotational speed, alongside contextual variables such as anomaly indicators, Remaining Useful Life (RUL), operational shift, and day of the week.

The Initial inspection showed No duplicate records were present.
Several sensor-related variables (pressure_bar, temp_celsius, flow_lpm, vibration_x_g, vibration_y_g, and pump_rpm) contained missing values, each with approximately 2,590 missing observations.
Timestamp-related variables were stored as object datatype rather than proper datetime format and would require conversion during preprocessing.
Binary variables such as is_anomaly and is_sensor_dropout were stored as integers and may later be converted into boolean or categorical types for improved interpretability.
The dataset combines continuous numerical variables with categorical operational features, making it suitable for both time-series analysis and predictive modeling.

Overall, the telemetry dataset provides a rich representation of machine behavior and forms the core foundation for failure prediction and Remaining Useful Life (RUL) estimation.

2. Failure Events Dataset (failure_df)

The failure_df dataset contains 9 failure event records across hydraulic machines and includes information such as failure timestamps, degradation start times, repair costs, downtime duration, and failure modes.

Initial findings include no duplicate or missing values were detected.
Timestamp variables (failure_timestamp and degradation_start_timestamp) were stored as object datatypes and will require conversion to datetime format.
The dataset contains structured failure labels that are essential for supervised learning tasks such as failure classification and RUL labeling.
The relatively small number of failure events suggests that the project may face class imbalance challenges during modeling.

This dataset serves as the primary source for identifying breakdown events and constructing predictive targets.

3. Maintenance Actions Dataset (maintenance_df)

The maintenance_df dataset contains 51 maintenance records documenting historical maintenance interventions performed on hydraulic machines. Features include maintenance timestamps, maintenance type, replaced components, technician identifiers, and associated maintenance costs.

Initial assessment revealed: No duplicate records were present.
One variable, component_replaced, contained 9 missing values, which may indicate maintenance activities where no component replacement occurred or where the information was not recorded.
Timestamp variables were stored as object datatype and require datetime conversion.
The dataset provides important operational context that can later be transformed into predictive features such as maintenance frequency, time since last maintenance, and maintenance history trends.

4. Equipment Metadata Dataset (equipment_df)

The equipment_df dataset contains 10 records representing static machine-level metadata for hydraulic equipment. Features include installation dates, total operating hours, fluid type, filter change history, and maintenance priority classification.

The Initial observations showed: No duplicate or missing values were detected. Date-related variables (installation_date and last_filter_change_date) were stored as object datatype and will require conversion to datetime format.
The dataset contains static machine characteristics that can enrich the telemetry dataset during merging and support machine-level analysis.